In [1]:
!pip install pymupdf sentence-transformers

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\vasu mittal\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import fitz
import os
import json
from sentence_transformers import SentenceTransformer


C:\Users\vasu mittal\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


load pdf


In [3]:
pdf_path = "digitalbook.pdf"
doc=fitz.open(pdf_path)
print ("Total pages:", len(doc))


Total pages: 565


In [4]:
pages = []
for page_number,page in enumerate(doc):
    text = page.get_text()
    pages.append({"page_number": page_number, "text": text})    
print(pages[0])  # Print the first page's content


{'page_number': 0, 'text': ''}


TEXT CLEANING

In [5]:
def clean_text(text):
    # Remove newlines and extra spaces
     text = text.replace('\n', ' ')
     text = ' '.join(text.split())
   
     return text

In [6]:
for page in pages:
    page['text'] = clean_text(page['text'])

print(pages[4])  # Print the cleaned first page's content




{'page_number': 4, 'text': 'Digital Design With an Introduction to the Verilog HDL FIFTH EDITION M. Morris Mano Emeritus Professor of Computer Engineering California State University, Los Angeles Michael D. Ciletti Emeritus Professor of Electrical and Computer Engineering University of Colorado at Colorado Springs Upper Saddle River Boston Columbus San Franciso New York Indianapolis London Toronto Sydney Singapore Tokyo Montreal Dubai Madrid Hong Kong Mexico City Munich Paris Amsterdam Cape Town'}


CHUNKING

In [7]:
def chunk_code(text, chunk_size=2000, overlap=200):
    """
    Splits the text into chunks of specified size with a specified overlap.
    
    Parameters:
    - text (str): The input text to be chunked.
    - chunk_size (int): The maximum size of each chunk.
    - overlap (int): The number of overlapping characters between chunks.
    
    Returns:
    - List[str]: A list of text chunks.
    """
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        start += chunk_size - overlap  # Move start forward by chunk_size minus overlap
    return chunks


In [8]:
chunks = []
for page in pages:
    page_chunks = chunk_code(page['text'])
    for chunk in page_chunks:
        chunks.append({"page_number": page['page_number'], "chunk": chunk})

print("tottal chunks:", len(chunks))
print(chunks[3])  # Print the first chunk's content

tottal chunks: 906
{'page_number': 5, 'chunk': "Library of Congress Cataloging-in-Publication Data Mano, M. Morris, 1927– Digital design : with an introduction to the verilog hdl / M. Morris Mano, Michael D. Ciletti.—5th ed. p. cm. Includes index. ISBN-13: 978-0-13-277420-8 ISBN-10: 0-13-277420-8 1. Electronic digital computers—Circuits. 2. Logic circuits. 3. Logic design. 4. Digital integrated circuits. I. Ciletti, Michael D. II. Title. TK7888.3.M343 2011 621.39'5—dc23 2011039094 Vice President and Editorial Director, ECS: Marcia J. Horton Executive Editor: Andrew Gilfillan Vice-President, Production: Vince O’Brien Executive Marketing Manager: Tim Galligan Marketing Assistant: Jon Bryant Permissions Project Manager: Karen Sanatar Senior Managing Editor: Scott Disanno Production Project Manager/Editorial Production Manager: Greg Dulles Cover Designer: Jayne Conte Cover Photo: Michael D. Ciletti Composition: Jouve India Private Limited Full-Service Project Management: Jouve India Privat

EMBEDDING

In [9]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1358.00it/s]


In [10]:
texts = [chunk['chunk'] for chunk in chunks]
embeddings = model.encode(texts,show_progress_bar=True)


Batches: 100%|██████████| 29/29 [00:38<00:00,  1.34s/it]


In [11]:
print("Embeddings shape:", embeddings.shape)  # Print the shape of the embeddings


Embeddings shape: (906, 384)


In [12]:
for i, chunk in enumerate(chunks):
    chunk['embedding'] = embeddings[i].tolist()  # Convert numpy array to list for JSON serialization

SAVING THE PROCESSED DATA

In [13]:
with open('chunks_with_embeddings.json', 'w') as f:
    json.dump(chunks, f, indent=4)

STORE EMBEDDING IN VECTORED DATABASE

In [14]:
!pip install faiss-cpu

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\vasu mittal\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [15]:
import faiss
import numpy as np

In [16]:
embedding_matrix = np.array([chunk['embedding'] for chunk in chunks]).astype('float32')
print("Embedding matrix shape:", embedding_matrix.shape)  # Print the shape of the embedding matrix

Embedding matrix shape: (906, 384)


In [17]:
dim = embedding_matrix.shape[1]  # Dimension of the embeddings
index = faiss.IndexFlatL2(dim)  # Create a FAISS index for L2 distance
index.add(embedding_matrix)  # Add the embeddings to the index  
print("Number of vectors in the index:", index.ntotal)  # Print the number of vectors in the index

Number of vectors in the index: 906


SEMANTIC SEARCH


In [18]:
def semantic_search(query, top_k=5):
    """
    Performs semantic search on the indexed embeddings using the provided query.
    
    Parameters:
    - query (str): The input query string.
    - top_k (int): The number of top results to return.
    
    Returns:
    - List[Dict]: A list of dictionaries containing the page number, chunk, and distance for each result.
    """
    query_embedding = model.encode([query]).astype('float32')  # Encode the query
    distances, indices = index.search(query_embedding, top_k)  # Search the index
    results = []
    for i in range(top_k):
        idx = indices[0][i]
        distance = distances[0][i]
        results.append({
            "page_number": chunks[idx]['page_number'],
            "chunk": chunks[idx]['chunk'],
            "distance": float(distance)
        })
    return results    

In [19]:
from sentence_transformers import SentenceTransformer

test_query = "What is the main topic of the book?"
# Ensure the embedding model is available before running semantic search
if 'model' not in globals():
    model = SentenceTransformer('all-MiniLM-L6-v2')

if 'index' not in globals():
    if 'embedding_matrix' not in globals():
        embedding_matrix = np.asarray(
            [chunk['embedding'] for chunk in chunks],
            dtype='float32'
        )
    index = faiss.IndexFlatL2(embedding_matrix.shape[1])
    index.add(embedding_matrix)

results = semantic_search(test_query, top_k=5)
for result in results:
    print(f"Page Number: {result['page_number']}, Distance: {result['distance']}")
    print(f"Chunk: {result['chunk']}\n")    

Page Number: 85, Distance: 1.298074722290039
Chunk: veral exercises and design problems based on Verilog throughout the book.

Page Number: 366, Distance: 1.576037049293518
Chunk: Design. Upper Saddle River, NJ: Prentice Hall.

Page Number: 128, Distance: 1.6590080261230469
Chunk: t is a comment (optional) providing useful information to the reader. The second line begins with the keyword module and starts the declaration (description) of the module; the last line completes the declaration with the keyword endmodule. The keyword module is followed by a name and a list of ports. The name ( Simple_Circuit in this example) is an identifier. Identifiers are names given to modules, variables (e.g., a signal), and other elements of the language so that they can be referenced in the design. In general, we choose meaningful names for modules. Identifiers are composed of alpha- numeric characters and the underscore (_), and are case sensitive. Identifiers must start with an alphabetic character

In [20]:
!pip install rank-bm25

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\vasu mittal\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [21]:
from rank_bm25 import BM25Okapi

In [22]:
corpus = [chunk['chunk'] for chunk in chunks]
bm25 = BM25Okapi([doc.split() for doc in corpus])
query = "What is the main topic of the book?"
scores = bm25.get_scores(query.split())
top_indices = scores.argsort()[-5:][::-1]  # Get indices of top 5 scores in descending order
for idx in top_indices:
    print(f"Page Number: {chunks[idx]['page_number']}, Score: {scores[idx]}")
    print(f"Chunk: {chunks[idx]['chunk']}\n")   

Page Number: 266, Score: 20.98471399179048
Chunk: Problems 249 (a)* Use D flip-flops in the design. (b) Use JK flip-flops in the design. 001 010 011 100 000 1/1 1/1 1/0 1/1 1/1 0/0 0/0 0/0 0/0 0/0 FIGURE P5.19 5.20 Design the sequential circuit specified by the state diagram of Fig. 5.19 , using T flip-flops. 5.21 What is the main difference between an initial statement and an always statement in Verilog HDL? 5.22 Draw the waveform generated by the statements below: (a) initial begin w = 0; #10 w = 1; # 40 w = 0; # 20 w = 1; #15 w = 0; end (b) initial fork w = 0; #10 w = 1; # 40 w = 0; # 20 w = 1; #15 w = 0; join 5.23* Consider the following statements assuming that RegA contains the value of 50 initially. (a) RegA = 125; (b) RegA <= 125; RegB = RegA; RegB <= RegA; What are the values of RegA and RegB after execution? 5.24 Write and verify an HDL behavioral description of a positive-edge-sensitive D flip-flop with asynchronous preset and clear. 5.25 A special positive-edge-triggered fl

In [23]:
# Map semantic results back to their chunk indices
semantic_rank = {}
for rank, item in enumerate(results, start=1):
    idx = next(
        i for i, chunk in enumerate(chunks)
        if chunk["page_number"] == item["page_number"]
        and chunk["chunk"] == item["chunk"]
    )
    semantic_rank[idx] = rank

bm25_rank = {int(idx): rank for rank, idx in enumerate(top_indices, start=1)}

# Reciprocal rank fusion; lower ranks are better
candidate_indices = set(semantic_rank) | set(bm25_rank)
final_scores = {
    idx: 1 / (60 + semantic_rank.get(idx, 1000))
       + 1 / (60 + bm25_rank.get(idx, 1000))
    for idx in candidate_indices
}

for idx in sorted(candidate_indices, key=final_scores.get, reverse=True)[:5]:
    print(f"Page Number: {chunks[idx]['page_number']}")
    print(f"Chunk: {chunks[idx]['chunk']}\n")

print("Final Scores:", final_scores)  # Print the final scores for debugging

Page Number: 85
Chunk: veral exercises and design problems based on Verilog throughout the book.

Page Number: 266
Chunk: Problems 249 (a)* Use D flip-flops in the design. (b) Use JK flip-flops in the design. 001 010 011 100 000 1/1 1/1 1/0 1/1 1/1 0/0 0/0 0/0 0/0 0/0 FIGURE P5.19 5.20 Design the sequential circuit specified by the state diagram of Fig. 5.19 , using T flip-flops. 5.21 What is the main difference between an initial statement and an always statement in Verilog HDL? 5.22 Draw the waveform generated by the statements below: (a) initial begin w = 0; #10 w = 1; # 40 w = 0; # 20 w = 1; #15 w = 0; end (b) initial fork w = 0; #10 w = 1; # 40 w = 0; # 20 w = 1; #15 w = 0; join 5.23* Consider the following statements assuming that RegA contains the value of 50 initially. (a) RegA = 125; (b) RegA <= 125; RegB = RegA; RegB <= RegA; What are the values of RegA and RegB after execution? 5.24 Write and verify an HDL behavioral description of a positive-edge-sensitive D flip-flop with 

In [24]:
from sentence_transformers import cross_encoder

# Initialize the cross-encoder for semantic rankingre
reranker = cross_encoder.CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 975.84it/s]


In [25]:
!pip install ollama
import ollama

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\vasu mittal\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [26]:
response = ollama.chat(
    model="llama3.1:8b",
    messages=[
        {
            "role": "user",
            "content": "Explain RAG in simple terms."
        }
    ]
)

print(response["message"]["content"])

RAG is an acronym that stands for "Red, Amber, Green". It's a color-coding system used to track progress, status, or priority levels, particularly in project management and IT fields.

Here's how it works:

1. **Red (R)**: Indicates a problem, issue, or critical failure. It's like a warning light that says "stop" or "attention needed".
2. **Amber (A)**: Represents a caution or a warning. It's like a yellow light that says "be careful" or "investigate further".
3. **Green (G)**: Signifies success, completion, or normal operation. It's like a green light that says "go" or "all clear".

RAG is often used to:

* Track project progress and identify potential issues
* Evaluate the status of tasks or systems
* Set priorities and focus on critical areas
* Communicate status updates to stakeholders

Think of RAG like a traffic light: Red means stop, Amber means caution, and Green means go!


In [27]:
import ollama

response = ollama.chat(
    model="llama3.1:8b",
    messages=[
        {
            "role": "user",
            "content": "In AI, what does RAG stand for? Explain Retrieval-Augmented Generation in 2 simple sentences."
        }
    ]
)

print(response["message"]["content"])

In AI, RAG stands for Retrieval-Augmented Generation.

Retrieval-Augmented Generation (RAG) is a type of AI model that combines the strengths of retrieval-based models and generation-based models. It uses a retrieval component to gather relevant information from a large database and then uses a generation component to create a new response or output by incorporating the retrieved information.


In [28]:
candidate_order = sorted(
    candidate_indices,
    key=final_scores.get,
    reverse=True
)[:10]

print("Number of candidates:", len(candidate_order))

pairs = [
    (test_query, chunks[idx]["chunk"])
    for idx in candidate_order
]

print("Pairs created:", len(pairs))

rerank_scores = reranker.predict(pairs)

reranked_results = sorted(
    zip(candidate_order, rerank_scores),
    key=lambda item: item[1],
    reverse=True
)

print("Reranking complete.")

for idx, score in reranked_results[:5]:
    print("Page:", chunks[idx]["page_number"])
    print("Reranker score:", float(score))
    print("Text:", chunks[idx]["chunk"][:500])
    print("-" * 80)

Number of candidates: 10
Pairs created: 10
Reranking complete.
Page: 11
Reranker score: -7.482226371765137
Text: x Preface (HDLs), and this edition of the book gives greater attention to their use and presents what we think is a clear development of a design methodology using the Verilog HDL. MULT I ‐M ODA L L E A R N IN G Digital Design supports a multimodal approach to learning. The so‐called VARK char- acterization of learning modalities identifies four major modes by which humans learn: (V) visual, (A) aural, (R) reading, and (K) kinesthetic. In hindsight, we note that the relatively high level of illu
--------------------------------------------------------------------------------
Page: 85
Reranker score: -8.129461288452148
Text: veral exercises and design problems based on Verilog throughout the book.
--------------------------------------------------------------------------------
Page: 129
Reranker score: -10.212762832641602
Text: 112 Chapter 3 Gate-Level Minimization The port l

In [29]:
def ask_documind(question, retrieval_k=5):

    # 1. Semantic search
    semantic_results = semantic_search(
        question,
        top_k=retrieval_k
    )

    semantic_rank = {}

    for rank, item in enumerate(semantic_results, start=1):
        idx = next(
            i for i, chunk in enumerate(chunks)
            if chunk["page_number"] == item["page_number"]
            and chunk["chunk"] == item["chunk"]
        )

        semantic_rank[idx] = rank

    # 2. BM25 keyword search
    tokenized_query = question.lower().split()

    bm25_scores = bm25.get_scores(tokenized_query)

    bm25_indices = bm25_scores.argsort()[-retrieval_k:][::-1]

    bm25_rank = {
        int(idx): rank
        for rank, idx in enumerate(bm25_indices, start=1)
    }

    # 3. Reciprocal Rank Fusion
    candidate_indices = set(semantic_rank) | set(bm25_rank)

    final_scores = {
        idx: (
            1 / (60 + semantic_rank.get(idx, 1000))
            + 1 / (60 + bm25_rank.get(idx, 1000))
        )
        for idx in candidate_indices
    }

    candidate_order = sorted(
        candidate_indices,
        key=final_scores.get,
        reverse=True
    )[:10]

    # 4. Neural reranking
    pairs = [
        (question, chunks[idx]["chunk"])
        for idx in candidate_order
    ]

    rerank_scores = reranker.predict(pairs)

    reranked_results = sorted(
        zip(candidate_order, rerank_scores),
        key=lambda item: item[1],
        reverse=True
    )

    # 5. Select the top 5 chunks
    top_results = reranked_results[:5]

    # 6. Build context with page metadata
    context_parts = []

    for idx, score in top_results:
        chunk = chunks[idx]

        context_parts.append(
            f"[Page {chunk['page_number']}]\n"
            f"{chunk['chunk']}"
        )

    context = "\n\n".join(context_parts)

    # 7. Create grounded prompt
    prompt = f"""
You are DocuMind, an AI document intelligence assistant.

Answer the user's question using ONLY the information provided in the context below.

If the answer is present in the context, answer it directly and clearly.
Do not say that the information is unavailable if the context contains the answer.
If the context genuinely does not contain the answer, then say:
"Information not found in the provided document."

DOCUMENT CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
"""

    # 8. Generate answer using Ollama
    response = ollama.chat(
        model="llama3.1:8b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        options={
            "num_ctx": 4096,
            "num_predict": 400,
            "temperature": 0.1
        },
        keep_alive="5m"
    )

    answer = response["message"]["content"]

    return {
        "question": question,
        "answer": answer,
        "retrieved_chunks": [
            {
                "chunk_id": idx,
                "page_number": chunks[idx]["page_number"],
                "reranker_score": float(score),
                "text": chunks[idx]["chunk"]
            }
            for idx, score in top_results
        ]
    }

In [32]:
result = ask_documind(
    "How does a D flip-flop work?"
)

print("QUESTION:")
print(result["question"])

print("\nANSWER:")
print(result["answer"])

print("\nSOURCES:")
for source in result["retrieved_chunks"]:
    print(
        f"Page: {source['page_number']} | "
        f"Reranker score: {source['reranker_score']:.4f}"
    )

QUESTION:
How does a D flip-flop work?

ANSWER:
A D flip-flop works by sampling the D input and changing its output Q only at the negative edge of the synchronizing or controlling clock (designated as Clk). When the clock is 0, the output of the inverter is 1, enabling the slave latch, and the value of Y is transferred to the output of the flip-flop at Q.

SOURCES:
Page: 217 | Reranker score: 5.6063
Page: 218 | Reranker score: 5.0505
Page: 217 | Reranker score: 4.8938
Page: 215 | Reranker score: 4.8636
Page: 516 | Reranker score: 4.5450
